# 🎼 Notebook 2 — Choreography vs Orchestration

Once you've decided to use a saga, you must decide **how the steps are coordinated**.
There are two classic styles:

### 🎯 Orchestration
A single **orchestrator** service calls each participant in order, tracks state, and decides
what to do on failure. Think of it as a conductor with a baton.

### 🎶 Choreography
Services **publish events** when they finish. Other services **subscribe** to the events they care about
and react. Nobody is in charge — the workflow is emergent, like dancers following the music.

Both can implement the **same saga**. The trade-off is about coupling, visibility, and who owns the flow.


## 🛠️ Setup

```bash
cd 05-microservices/saga
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook).
If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 1. 🎯 Orchestration — a central coordinator

The orchestrator owns the saga state machine. Every participant has two HTTP endpoints
(or two message handlers): the forward action and the compensating action.
In real life the orchestrator would be a service like **Temporal**, **Camunda**, or **AWS Step Functions**.
Here we'll implement it as a small Python class so you can see exactly what it does.


In [1]:
# Simulated services (in-memory)
inventory = {"widget": 10}
payments  = {"charged": 0, "refunds": 0}
shipping  = {"bookings": []}

class InventoryService:
    def reserve(self, item): inventory[item] -= 1;     print(f"  📦 reserved {item}")
    def release(self, item): inventory[item] += 1;     print(f"  📦 released {item}")

class PaymentService:
    def charge(self, amt):   payments["charged"] += amt; print(f"  💳 charged ${amt}")
    def refund(self, amt):   payments["refunds"] += amt; print(f"  💳 refunded ${amt}")

class ShippingService:
    def __init__(self, fail=False): self.fail = fail
    def book(self, item):
        if self.fail: raise RuntimeError("courier API timeout")
        shipping["bookings"].append(item); print(f"  🚚 booked {item}")
    def cancel(self, item):
        if item in shipping["bookings"]: shipping["bookings"].remove(item)
        print(f"  🚚 cancelled {item}")

inv, pay, ship = InventoryService(), PaymentService(), ShippingService(fail=True)


In [2]:
class CheckoutOrchestrator:
    """Orchestrator owns the flow. It calls each service and tracks compensations."""
    def __init__(self, inv, pay, ship):
        self.inv, self.pay, self.ship = inv, pay, ship

    def checkout(self, item, amount):
        print(f"🎯 orchestrator: start checkout({item}, ${amount})")
        compensations = []
        try:
            self.inv.reserve(item);          compensations.append(lambda: self.inv.release(item))
            self.pay.charge(amount);         compensations.append(lambda: self.pay.refund(amount))
            self.ship.book(item);            compensations.append(lambda: self.ship.cancel(item))
            print("🎯 orchestrator: DONE")
            return "committed"
        except Exception as e:
            print(f"🎯 orchestrator: failure — {e}; compensating...")
            for c in reversed(compensations):
                c()
            return "compensated"

result = CheckoutOrchestrator(inv, pay, ship).checkout("widget", 20)
print("\nresult:", result, "| inventory:", inventory, "| payments:", payments)


🎯 orchestrator: start checkout(widget, $20)
  📦 reserved widget
  💳 charged $20
🎯 orchestrator: failure — courier API timeout; compensating...
  💳 refunded $20
  📦 released widget

result: compensated | inventory: {'widget': 10} | payments: {'charged': 20, 'refunds': 20}


### ✅ What's nice about orchestration
- The **entire workflow lives in one place**. You can draw the state machine on a whiteboard.
- Easy to **debug, monitor, and change order** of steps.
- Error handling is explicit.

### ⚠️ What hurts
- The orchestrator is a **central component** — if it's down, nothing moves.
- It **knows about every participant** (tight coupling from its side).
- Tempting to put business rules in it that really belong in the participants.


## 2. 🎶 Choreography — services react to events

No orchestrator. Each service publishes events ("`OrderPlaced`", "`StockReserved`", "`PaymentCharged`", ...)
and each service decides which events it cares about.
In real life this is a message broker like **Kafka**, **RabbitMQ**, **NATS**, or **AWS SNS/SQS**.
Here's a ~10-line in-memory bus so we can watch it happen.


In [3]:
from collections import defaultdict

class EventBus:
    def __init__(self): self.subs = defaultdict(list)
    def on(self, event, handler): self.subs[event].append(handler)
    def emit(self, event, **data):
        print(f"  📨 event: {event} {data}")
        for h in list(self.subs[event]):
            h(**data)

# fresh "databases"
inventory = {"widget": 10}
payments  = {"charged": 0, "refunds": 0}
shipping  = {"bookings": []}
bus = EventBus()

# --- inventory service ---
def on_order_placed(order_id, item, amount):
    inventory[item] -= 1
    bus.emit("StockReserved", order_id=order_id, item=item, amount=amount)

def on_payment_failed(order_id, item, **_):
    inventory[item] += 1
    print(f"  📦 compensated: released {item}")

bus.on("OrderPlaced",    on_order_placed)
bus.on("PaymentFailed",  on_payment_failed)

# --- payment service (we'll make it fail to show compensation) ---
def on_stock_reserved(order_id, item, amount):
    # 💥 simulate payment failure
    print("  💳 payment refused: insufficient funds")
    bus.emit("PaymentFailed", order_id=order_id, item=item, amount=amount)

bus.on("StockReserved", on_stock_reserved)

# --- shipping service ---
def on_payment_ok(order_id, item, **_):
    shipping["bookings"].append(item)
    bus.emit("Shipped", order_id=order_id, item=item)
bus.on("PaymentOK", on_payment_ok)

# kick it off
bus.emit("OrderPlaced", order_id=1, item="widget", amount=20)

print()
print("inventory:", inventory, "| payments:", payments, "| shipping:", shipping)


  📨 event: OrderPlaced {'order_id': 1, 'item': 'widget', 'amount': 20}
  📨 event: StockReserved {'order_id': 1, 'item': 'widget', 'amount': 20}
  💳 payment refused: insufficient funds
  📨 event: PaymentFailed {'order_id': 1, 'item': 'widget', 'amount': 20}
  📦 compensated: released widget

inventory: {'widget': 10} | payments: {'charged': 0, 'refunds': 0} | shipping: {'bookings': []}


Notice what just happened:

1. `OrderPlaced` → inventory reserves stock → emits `StockReserved`.
2. `StockReserved` → payment refuses → emits `PaymentFailed`.
3. `PaymentFailed` → inventory releases stock (**its own compensation**).

There is **no central checkout service**. The saga is the *set of subscriptions*.

### ✅ What's nice
- Services are **loosely coupled** — they just know the event shapes.
- Easy to add a new participant: subscribe and go.

### ⚠️ What hurts
- **No single place** shows the full flow. You read 4 services to understand what happens.
- Easy to create **event cycles** by accident (A triggers B triggers A…).
- Each service must implement *its own* compensation logic, which is easy to miss.


## 3. Side-by-side cheat sheet

| Concern                     | Orchestration                         | Choreography                           |
|-----------------------------|----------------------------------------|-----------------------------------------|
| Where is the flow?          | One orchestrator                       | Spread across services                  |
| Coupling                    | Orchestrator knows all participants    | Participants know event shapes          |
| Visibility / debugging      | Easy — one log                         | Hard — trace events across services     |
| Good fit                    | Complex, long-running workflows        | Few steps, highly decoupled teams       |
| Risk                        | Orchestrator becomes a "god" service   | Workflow becomes implicit spaghetti     |
| Real-world tools            | Temporal, Camunda, AWS Step Functions  | Kafka, RabbitMQ, NATS, AWS SNS/SQS      |

### Rule of thumb
- **Few steps, well-defined teams, independent domains** → choreography.
- **Many steps, complex branching, or you need a dashboard** → orchestration.
- When in doubt, start **orchestrated**: it's easier to reason about and you can always break it apart later.
